# Graphe de Synergies
**Objectif :** construire un graphe NetworkX où :
- Chaque **nœud** = une carte
- Chaque **arête** = une synergie mesurée (Jaccard >= 0.3, count >= 5)
- Le **poids** de l'arête = score Jaccard

Ce graphe permet d'identifier les cartes centrales d'un archetype,
les ponts entre archetypes, et l'impact d'un futur ban.

In [ ]:
import sqlite3
import pandas as pd
import networkx as nx

con = sqlite3.connect('../data/yugioh.db')

# Charger les paires significatives
# Filtre : Jaccard >= 0.3 ET au moins 5 decks en commun
pairs = pd.read_sql("""
    SELECT card_a, card_b, jaccard, cooc_count
    FROM card_cooccurrence
    WHERE jaccard >= 0.3
    AND cooc_count >= 5
    ORDER BY jaccard DESC
""", con)

print(f'Paires chargées : {len(pairs):,}')

## 1. Construction du graphe

In [ ]:
G = nx.Graph()

for _, row in pairs.iterrows():
    G.add_edge(row['card_a'], row['card_b'],
               weight=row['jaccard'],
               count=row['cooc_count'])

print(f'Noeuds (cartes) : {G.number_of_nodes():,}')
print(f'Aretes (synergies) : {G.number_of_edges():,}')
print(f'Composantes connexes : {nx.number_connected_components(G):,}')

## 2. Centralite — quelles cartes sont les plus connectees ?

In [ ]:
degree = pd.Series(dict(G.degree(weight='weight')), name='weighted_degree')
degree = degree.sort_values(ascending=False)

print('Top 20 cartes les plus connectees :')
degree.head(20)

## 3. Communautes — detection automatique des archetypes

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G, weight='weight'))
print(f'{len(communities)} communautes detectees')
print()

communities_sorted = sorted(communities, key=len, reverse=True)
for i, comm in enumerate(communities_sorted[:8]):
    cards_list = sorted(comm)[:6]
    print(f'Communaute {i+1} ({len(comm)} cartes) : {", ".join(cards_list)}...')

## 4. Focus sur un archetype — sous-graphe Maliss

In [ ]:
def archetype_subgraph(G, keyword, min_jaccard=0.3):
    nodes = [n for n in G.nodes() if keyword.lower() in n.lower()]
    neighbors = set(nodes)
    for n in nodes:
        for neighbor, data in G[n].items():
            if data['weight'] >= min_jaccard:
                neighbors.add(neighbor)
    return G.subgraph(neighbors)

maliss_sg = archetype_subgraph(G, 'Maliss')
print(f'Sous-graphe Maliss : {maliss_sg.number_of_nodes()} cartes, {maliss_sg.number_of_edges()} synergies')
print()

maliss_degree = pd.Series(dict(maliss_sg.degree(weight='weight'))).sort_values(ascending=False)
print('Cartes les plus centrales dans Maliss :')
maliss_degree.head(10)

## 5. Simulation de ban — impact sur le graphe

In [ ]:
def simulate_ban(G, card_name):
    if card_name not in G:
        print(f'Carte "{card_name}" introuvable dans le graphe.')
        return
    
    neighbors = list(G.neighbors(card_name))
    weights = [G[card_name][n]['weight'] for n in neighbors]
    impact = pd.Series(weights, index=neighbors).sort_values(ascending=False)
    
    print(f'Ban de "{card_name}"')
    print(f'  Connexions supprimees : {len(neighbors)}')
    print(f'  Cartes les plus impactees :')
    for card, w in impact.head(10).items():
        print(f'    {w:.2f}  {card}')

simulate_ban(G, 'Ash Blossom & Joyous Spring')

In [ ]:
simulate_ban(G, 'Maliss P March Hare')